In [ ]:
# after extract all the surface patches from step1
# In step 2, we perform all the prediction
# condition: 1 use all the positive patches 2 negative patch residue1 SasaRate > 0.2 and residue 2 SasaRate > 0.2

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages_CNN', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages_CNN')

In [ ]:
import os
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
import random

import itertools
import math

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import TensorDataset, Dataset, DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
save_dir = './content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test_processed_step1_with_SI/'

In [ ]:
#@title load cnn model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(    # input shape (1,15,20)
            nn.Conv2d(
                in_channels=1,         # input height: layers
                out_channels=16,       # n fileters
                kernel_size=(3,3),         # filter size
                stride=1,              # filter movement/step
                padding=(1,1),             # want same width and length of this image after Conv2d
            ),                         # ==> (16, 15, 20)
            nn.BatchNorm2d(16),        #applying batch norm
            nn.ReLU(),                 # activation
            nn.MaxPool2d(kernel_size=(3,2)), # choose max value in 2x2 area, output shape
        )                              # output shape (16,5,10)
        self.conv2 = nn.Sequential(    # input shape (16,5,10)
            nn.Conv2d(16,32,(2,2),1,(1,1)),    # output shape(32,5,10)
            nn.BatchNorm2d(32),        #applying batch norm
            nn.ReLU(),                 # activation
            nn.MaxPool2d(2),           # output shape (32,3,5)
        )
        self.fc1 = nn.Linear(32 * 3 * 5, 128)
        self.fc2 = nn.Linear(128, 32)
        self.out = nn.Linear(32, 1) # fully connected layer, output 10 classes

    def forward(self,x):
        x = self.conv1(x)
        x = self.conv2(x)              # (batch, 32, 7, 7)
        x = x.view(x.size(0), -1)      # flatten the output of conv2 to (batch_size, 32*7*7)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        output = torch.sigmoid(self.out(x))
        return output

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_cnn = CNN().to(device)

In [ ]:
#@title perform prediction

# 1. Load the trained model
model_dir = './drive/MyDrive/@antigen_epitope/abdb_v2/data_processed_step1_with_SI/'
model_path = os.path.join(model_dir, 'model_cnn.pth')
model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
model_cnn = model_cnn.to(device)
model_cnn.eval()  # Set the model to evaluation mode

BATCH_SIZE = 500

# 2. Convert the testing patches into a DataLoader
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_processed_step1_with_SI/yuhao/'
testing_patches = np.load(f'{save_dir}cnn_testing_patch_3points.npy')
testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)

# Create a DataLoader for the testing patches (assuming no labels as this is for testing)
testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

# 3. Loop through the DataLoader and pass each batch to the model
predictions = []
with torch.no_grad():  # Deactivate autograd engine to reduce memory usage and speed up computations
    for batch in testing_loader:
        batch = batch.to(device)
        outputs = model_cnn(batch)
        outputs = outputs.squeeze().cpu().numpy()
        predictions.extend(outputs)

# 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
binary_predictions = predictions
#binary_predictions = [1 if pred >= 0.5 else 0 for pred in predictions]
print(binary_predictions)

# If you want to save predictions to a file
np.save(f'{save_dir}predictions.npy', np.array(binary_predictions))

In [ ]:
len(binary_predictions)

In [ ]:
#@title get patch residue index from txt file

import pandas as pd

# File paths
input_file_path = "/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_processed_step1_with_SI/yuhao/cnn_testing_patch_3points.txt"
output_file_path = "/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_processed_step1_with_SI/yuhao/testing_patch_index.csv"

# Lists to store the data
protein_names = []
patch_indices = []

# Open and read the text file
with open(input_file_path, 'r') as file:
    lines = file.readlines()
    current_protein_name = None
    for line in lines:
        line = line.strip()  # Remove whitespace and newline characters
        if line.startswith('Patches for protein'):
            current_protein_name = line.split(' ')[3][:-1]  # Extract the protein name
        elif line.startswith('indices:'):
            # Extract the patch indices
            indices = line.split('[')[1].split(']')[0]
            protein_names.append(current_protein_name)
            patch_indices.append(indices)

# Create a DataFrame
df_patch = pd.DataFrame({
    'Protein_Name': protein_names,
    'Patch_Indices': patch_indices
})

# Save the DataFrame to a CSV file
df_patch.to_csv(output_file_path, index=False)

# Display the first few rows of the dataframe
print(df_patch.head())
df_patch.shape

In [ ]:
df_patch['Predictions'] = binary_predictions

In [ ]:
df_patch.head()

In [ ]:
#@title Read the original data frame
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test/'
filtered_combined_df = pd.read_csv(os.path.join(save_dir, 'all_data_with_SI.csv'))

In [ ]:
#@title merge the data frame with patch predictions
# Step 1: Transform df_patch to long format
df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

# Step 2: Merge on Protein_Name and Patch_Indices
merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

# Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
merged_df['prediction'] = merged_df['Predictions']
merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)

# If you want NaN for residues without a prediction, this step is optional
merged_df['prediction'].fillna(np.nan, inplace=True)

# Normalize the prediction values for each protein
merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
merged_df.drop('prediction', axis=1, inplace=True)

df_groupby_cols = merged_df.columns.tolist()[:-1]
merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()

merged_df.to_csv(os.path.join(save_dir, 'all_data_long_pred.csv'))
merged_df.head()

In [ ]:
merged_df

In [ ]:
# Filter out rows where 'pred' is NaN
merged_df_filtered = merged_df.dropna(subset=['pred'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming merged_df is your DataFrame

# Set up the matplotlib figure
plt.figure(figsize=(10, 6))

# Plot the distribution of pred values for each Status
sns.histplot(data=merged_df_filtered, x='pred', hue='Status', element='step', stat="density", common_norm=False)

# Provide labels and title
plt.xlabel('Prediction Value')
plt.ylabel('Density')
plt.title('Distribution of Prediction Values by Status')
plt.legend(title='Status')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get true binary labels
true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

# Get prediction scores
pred_scores = merged_df_filtered['pred'].values

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(true_labels, pred_scores)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from scipy.stats import spearmanr

# Get true binary labels
true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

# Get prediction scores
pred_scores = merged_df_filtered['pred'].values

# Convert prediction scores to binary predictions (assuming a threshold of 0.5)
pred_labels = np.where(pred_scores >= 0.5, 1, 0)

# Calculate Matthews Correlation Coefficient (MCC)
mcc = matthews_corrcoef(true_labels, pred_labels)
print("Matthews Correlation Coefficient (MCC):", mcc)

# Calculate Spearman correlation coefficient
spearman_corr, _ = spearmanr(true_labels, pred_scores)
print("Spearman Correlation Coefficient:", spearman_corr)

# Calculate ROC AUC score
roc_auc = roc_auc_score(true_labels, pred_scores)
print("ROC AUC Score:", roc_auc)

In [ ]:
#@title find the best cut-off
from sklearn.metrics import roc_curve

# Convert 'Status' column to binary values
true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

# Get false positive rate, true positive rate and thresholds using roc_curve
fpr, tpr, thresholds = roc_curve(true_labels, merged_df_filtered['pred'])

# Calculate Youden's J statistic for each threshold
J = tpr - fpr

# Get the index of the maximum J value
idx = np.argmax(J)

# Get the best threshold
best_threshold = thresholds[idx]

print(f"Best cut-off (threshold): {best_threshold:.3f}")


In [ ]:
# Convert 'Status' column to binary values
true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

# Threshold 'pred' values at 0.5 to get predicted classes
predicted_labels = np.where(merged_df_filtered['pred'] >= 0.5, 1, 0)

# Calculate accuracy
accuracy = np.mean(true_labels == predicted_labels)

print(f"Accuracy: {accuracy*100:.2f}%")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Convert 'Status' column to binary values
true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

# Calculate metrics
accuracy = accuracy_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
precision = precision_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
recall = recall_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
f1 = f1_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
auc = roc_auc_score(true_labels, merged_df_filtered['pred'])

# Display metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Best Cutoff (Threshold): {best_threshold:.3f}")
